# Live Skill — Notebook Control

SAPIEN 데스크탑 뷰어 창을 열어 시각 확인. 노트북 안에 inline 표시 안 됨 — **별도 OS 창**으로 뜸.

기본 호출은 **subprocess로 백그라운드 spawn** → 노트북 셀은 즉시 반환, 창은 별도 프로세스로 떠있음. 닫고 싶으면 `Popen.terminate()` 호출 또는 창 X 버튼.

텔레오퍼레이션(직접 조작)은 이 라운드에 미포함 (pinocchio + IK 필요).


## Setup

In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

TRAJ = (PROJECT_ROOT / 'data' / 'datasets' / 'PickCube-v1'
        / 'trajectory.rgb.pd_joint_pos.physx_cpu.h5')
print('replay traj:', TRAJ.name, '| exists:', TRAJ.exists())

## ① browse(task) — 환경만 띄움 (자동 검증: spawn 후 2초 뒤 종료)

실행 시 SAPIEN 창이 잠시 화면에 나타났다 사라집니다.

In [ ]:
from scripts.live import browse

p = browse(task='PickCube-v1')
print(f'spawned pid={p.pid}')
time.sleep(2)
print('alive after 2s:', p.poll() is None)
p.terminate()
try: p.wait(timeout=5)
except Exception: p.kill()
print('exit code:', p.returncode)

## ② replay(traj_path, episode) — 저장된 에피소드를 창에서 재생

에피소드 0의 모션 플래닝 동작이 창에 뜹니다. 같은 자동 종료 패턴.

In [ ]:
from scripts.live import replay

p = replay(traj_path=TRAJ, episode=0)
print(f'spawned pid={p.pid}')
time.sleep(3)
print('alive after 3s:', p.poll() is None)
p.terminate()
try: p.wait(timeout=5)
except Exception: p.kill()
print('exit code:', p.returncode)

## ③ 자동 종료 없이 직접 닫기 — 사용 패턴

다음 셀은 직접 실행 시 SAPIEN 창이 뜨고 **사용자가 X 버튼으로 닫을 때까지** 유지됩니다. nbconvert 자동 실행에서 무한 대기를 피하려고 `if False:` 가드를 걸어둠 — 노트북에서 직접 쓸 때 `True`로 바꾸세요.

In [ ]:
if False:
    p = browse(task='PickCube-v1')
    print('Window open. Close it to free the subprocess.')
    p.wait()
    print('closed by user, exit code:', p.returncode)